# 02 - Target Construction

This notebook implements Stage 2 of the credit-risk pipeline.

It builds borrower-time snapshots and constructs two labels:
- `missed_upcoming_emi`
- `future_dpd30`

The notebook uses cleaned artifacts from Stage 1 and saves target tables to `artifacts/processed/`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PROCESSED_DIR = ROOT / 'artifacts' / 'processed'
RAW_CHECKS_DIR = ROOT / 'artifacts' / 'raw_checks'

required = [
    PROCESSED_DIR / 'cleaned_installment_events.parquet',
    PROCESSED_DIR / 'cleaned_pos_cash_balance.parquet',
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f'Missing required cleaned artifact: {path}')

print('Stage 2 inputs verified.')


In [ ]:
inst = pd.read_parquet(PROCESSED_DIR / 'cleaned_installment_events.parquet')
pos = pd.read_parquet(PROCESSED_DIR / 'cleaned_pos_cash_balance.parquet')

print('installment events shape:', inst.shape)
print('pos shape:', pos.shape)
display(inst.head())
display(pos.head())


## Label Rules

### Snapshot definition
Each snapshot is anchored on an observed installment event for a borrower.

Snapshot time `T`:
- `snapshot_day = DAYS_INSTALMENT` of the current installment event

### `missed_upcoming_emi`
For the next installment event after `T` for the same borrower:
- label `1` if the next installment is paid after due date, missing, or still underpaid after aggregation
- label `0` otherwise

### `future_dpd30`
Within the next `90` days after `T`:
- label `1` if future installment events or POS records indicate `30+` days past due
- label `0` otherwise

Rows without a complete future window or without a next installment are excluded from the corresponding target.


In [ ]:
inst = inst.sort_values(['SK_ID_CURR', 'DAYS_INSTALMENT', 'NUM_INSTALMENT_NUMBER', 'NUM_INSTALMENT_VERSION']).reset_index(drop=True)
inst['snapshot_day'] = inst['DAYS_INSTALMENT']
inst['snapshot_month_approx'] = np.floor(inst['snapshot_day'] / 30.0).astype('int32')

borrower_event_order = inst.groupby('SK_ID_CURR').cumcount()
inst['borrower_event_order'] = borrower_event_order.astype('int32')

next_cols = [
    'SK_ID_PREV', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER', 'DAYS_INSTALMENT',
    'first_payment_day', 'AMT_INSTALMENT', 'total_amount_paid', 'missing_payment_flag',
    'underpaid_after_aggregation_flag', 'aggregated_payment_delay_days'
]
next_events = (
    inst.groupby('SK_ID_CURR')[next_cols]
    .shift(-1)
    .add_prefix('next_')
)

snapshots = pd.concat([inst.copy(), next_events], axis=1)
snapshots['has_next_installment'] = snapshots['next_DAYS_INSTALMENT'].notna().astype('int8')
snapshots['next_installment_gap_days'] = snapshots['next_DAYS_INSTALMENT'] - snapshots['snapshot_day']

print('Snapshot base rows:', len(snapshots))
display(snapshots.head())


In [ ]:
missed_mask = snapshots['has_next_installment'] == 1
missed = snapshots.loc[missed_mask].copy()

missed['missed_upcoming_emi'] = (
    (missed['next_missing_payment_flag'].fillna(0) > 0)
    | (missed['next_underpaid_after_aggregation_flag'].fillna(0) > 0)
    | (missed['next_aggregated_payment_delay_days'].fillna(-999999) > 0)
).astype('int8')

missed['target_available_flag'] = 1
missed['target_name'] = 'missed_upcoming_emi'
missed['snapshot_id'] = (
    missed['SK_ID_CURR'].astype('int64').astype(str)
    + '_'
    + missed['borrower_event_order'].astype(str)
)

missed_cols = [
    'snapshot_id', 'SK_ID_CURR', 'SK_ID_PREV', 'borrower_event_order', 'snapshot_day',
    'snapshot_month_approx', 'DAYS_INSTALMENT', 'first_payment_day', 'aggregated_payment_delay_days',
    'missing_payment_flag', 'underpaid_after_aggregation_flag', 'next_SK_ID_PREV',
    'next_NUM_INSTALMENT_NUMBER', 'next_DAYS_INSTALMENT', 'next_first_payment_day',
    'next_aggregated_payment_delay_days', 'next_missing_payment_flag',
    'next_underpaid_after_aggregation_flag', 'next_installment_gap_days',
    'missed_upcoming_emi', 'target_available_flag', 'target_name'
]
missed_target = missed[missed_cols].sort_values(['SK_ID_CURR', 'snapshot_day']).reset_index(drop=True)

print(f"Built missed_upcoming_emi target with {len(missed_target):,} rows.")
print(f"Positive rate: {missed_target['missed_upcoming_emi'].mean():.4f}")

display(missed_target.head())
display(missed_target['missed_upcoming_emi'].value_counts(dropna=False).rename_axis('missed_upcoming_emi').to_frame('count'))


In [ ]:
future_window_days = 90

inst_future = inst[['SK_ID_CURR', 'DAYS_INSTALMENT', 'aggregated_payment_delay_days']].copy()
inst_future['inst_dpd30_flag'] = (inst_future['aggregated_payment_delay_days'].fillna(-999999) >= 30).astype('int8')

pos = pos.copy()
pos['approx_day_from_month'] = pos['MONTHS_BALANCE'] * 30
pos['pos_dpd30_flag'] = ((pos['SK_DPD'].fillna(0) >= 30) | (pos['SK_DPD_DEF'].fillna(0) >= 30)).astype('int8')

borrower_max_future_day = inst.groupby('SK_ID_CURR')['DAYS_INSTALMENT'].max().rename('borrower_max_installment_day')
snapshots = snapshots.merge(borrower_max_future_day, on='SK_ID_CURR', how='left')
snapshots['future_window_complete_flag'] = (
    snapshots['borrower_max_installment_day'] >= (snapshots['snapshot_day'] + future_window_days)
).astype('int8')

def build_flagged_day_dict(df: pd.DataFrame, borrower_col: str, day_col: str, flag_col: str) -> dict[int, np.ndarray]:
    flagged = df.loc[df[flag_col] == 1, [borrower_col, day_col]].copy()
    if flagged.empty:
        return {}
    flagged = flagged.sort_values([borrower_col, day_col])
    return {
        int(borrower_id): group[day_col].to_numpy(dtype='float64', copy=True)
        for borrower_id, group in flagged.groupby(borrower_col, sort=False)
    }

inst_dpd30_days = build_flagged_day_dict(inst_future, 'SK_ID_CURR', 'DAYS_INSTALMENT', 'inst_dpd30_flag')
pos_dpd30_days = build_flagged_day_dict(pos, 'SK_ID_CURR', 'approx_day_from_month', 'pos_dpd30_flag')

print(f'Borrowers with installment 30+ DPD hits: {len(inst_dpd30_days):,}')
print(f'Borrowers with POS 30+ DPD hits: {len(pos_dpd30_days):,}')

snapshot_groups = snapshots.groupby('SK_ID_CURR', sort=False).indices
total_borrowers = len(snapshot_groups)
inst_hits = np.zeros(len(snapshots), dtype=np.int8)
pos_hits = np.zeros(len(snapshots), dtype=np.int8)
start_time = time.time()
print(f'Starting future_dpd30 construction for {total_borrowers:,} borrowers...')

for idx, (borrower_id, row_idx) in enumerate(snapshot_groups.items(), start=1):
    borrower_snapshot_days = snapshots.iloc[row_idx]['snapshot_day'].to_numpy(dtype='float64', copy=False)
    window_end_days = borrower_snapshot_days + future_window_days

    inst_days = inst_dpd30_days.get(int(borrower_id))
    if inst_days is not None and len(inst_days) > 0:
        left = np.searchsorted(inst_days, borrower_snapshot_days, side='right')
        right = np.searchsorted(inst_days, window_end_days, side='right')
        inst_hits[row_idx] = (right > left).astype(np.int8)

    pos_days = pos_dpd30_days.get(int(borrower_id))
    if pos_days is not None and len(pos_days) > 0:
        left = np.searchsorted(pos_days, borrower_snapshot_days, side='right')
        right = np.searchsorted(pos_days, window_end_days, side='right')
        pos_hits[row_idx] = (right > left).astype(np.int8)

    if idx == 1 or idx % 5000 == 0 or idx == total_borrowers:
        elapsed = time.time() - start_time
        print(f'Processed {idx:,}/{total_borrowers:,} borrowers in {elapsed:.1f}s')

snapshots['future_inst_dpd30_hit'] = inst_hits
snapshots['future_pos_dpd30_hit'] = pos_hits
snapshots['future_dpd30'] = (
    (snapshots['future_inst_dpd30_hit'] > 0)
    | (snapshots['future_pos_dpd30_hit'] > 0)
).astype('int8')

future_dpd30_target = snapshots.loc[snapshots['future_window_complete_flag'] == 1].copy()
future_dpd30_target['target_available_flag'] = 1
future_dpd30_target['target_name'] = 'future_dpd30'
future_dpd30_target['snapshot_id'] = (
    future_dpd30_target['SK_ID_CURR'].astype('int64').astype(str)
    + '_'
    + future_dpd30_target['borrower_event_order'].astype(str)
)

future_cols = [
    'snapshot_id', 'SK_ID_CURR', 'SK_ID_PREV', 'borrower_event_order', 'snapshot_day',
    'snapshot_month_approx', 'borrower_max_installment_day', 'future_window_complete_flag',
    'future_inst_dpd30_hit', 'future_pos_dpd30_hit', 'future_dpd30', 'target_available_flag', 'target_name'
]
future_dpd30_target = future_dpd30_target[future_cols].sort_values(['SK_ID_CURR', 'snapshot_day']).reset_index(drop=True)

print(f"Built future_dpd30 target with {len(future_dpd30_target):,} rows.")
print(f"Positive rate: {future_dpd30_target['future_dpd30'].mean():.4f}")

display(future_dpd30_target.head())
display(future_dpd30_target['future_dpd30'].value_counts(dropna=False).rename_axis('future_dpd30').to_frame('count'))


In [ ]:
missed_target_path = PROCESSED_DIR / 'target_missed_upcoming_emi.parquet'
future_target_path = PROCESSED_DIR / 'target_future_dpd30.parquet'
label_summary_path = RAW_CHECKS_DIR / 'stage2_target_summary.csv'
label_notes_path = RAW_CHECKS_DIR / 'stage2_target_notes.md'

missed_target.to_parquet(missed_target_path, index=False)
future_dpd30_target.to_parquet(future_target_path, index=False)

summary = pd.DataFrame([
    {
        'target_name': 'missed_upcoming_emi',
        'row_count': len(missed_target),
        'positive_count': int(missed_target['missed_upcoming_emi'].sum()),
        'positive_rate': float(missed_target['missed_upcoming_emi'].mean()),
    },
    {
        'target_name': 'future_dpd30',
        'row_count': len(future_dpd30_target),
        'positive_count': int(future_dpd30_target['future_dpd30'].sum()),
        'positive_rate': float(future_dpd30_target['future_dpd30'].mean()),
    },
])
summary.to_csv(label_summary_path, index=False)

notes = '\n'.join([
    '# Stage 2 Target Notes',
    '',
    '- Snapshot anchor: observed borrower installment event at `snapshot_day = DAYS_INSTALMENT`.',
    '- `missed_upcoming_emi` uses the next borrower installment event after the snapshot.',
    '- `future_dpd30` uses a fixed 90-day future window.',
    '- `future_dpd30` is triggered by either installment delay >= 30 days or POS DPD >= 30 in the window.',
    '- Incomplete future windows are excluded from `future_dpd30`.',
])
label_notes_path.write_text(notes + '\n', encoding='utf-8')

manifest = {
    'targets': [missed_target_path.name, future_target_path.name],
    'summary': label_summary_path.name,
    'notes': label_notes_path.name,
    'future_window_days': future_window_days,
}
(RAW_CHECKS_DIR / 'stage2_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print(f'Saved: {missed_target_path}')
print(f'Saved: {future_target_path}')
print(f'Saved: {label_summary_path}')
print(f'Saved: {label_notes_path}')
display(summary)


## Stage 2 Outputs

Saved artifacts:
- `artifacts/processed/target_missed_upcoming_emi.parquet`
- `artifacts/processed/target_future_dpd30.parquet`
- `artifacts/raw_checks/stage2_target_summary.csv`
- `artifacts/raw_checks/stage2_target_notes.md`
- `artifacts/raw_checks/stage2_manifest.json`

These target tables will feed Stage 3 feature engineering and dataset assembly.
